# FastAPI Routing: Path and Query Parameters
## Week 4 · Member 6 · Harbour learning programme

**Aim:** Define routes with path and query parameters, and explain the difference.

**Friday task:** Implement book and loan CRUD endpoints for the **Library Loans API**.

This is a **30-cell** beginner lesson. It includes runnable routes, successful and failed requests, Uvicorn instructions, exercises and 20 teaching questions. Allow about 90 minutes; use the independent build task for a longer session.

**Outcome:** Given a request such as `GET /books/7?include_details=true`, you can identify the method, route, path parameter and query parameter, then write a matching handler.

The demo uses invented data and in-memory dictionaries. It is a routing prototype, not the finished PostgreSQL application.

## 1. Where your work fits

The [Week 4 README](https://github.com/Snrboi/harbour/blob/learn/README.md#week-4-fastapi-validation-tests-docker-and-ci-mini-project-1) assigns:
- **Member 6 — you:** routing, path/query parameters, Uvicorn; book and loan CRUD.
- **Member 1:** router/service/repository separation and borrowing rules.
- **Member 7:** full Pydantic validation, ISBN and due-date rules.
- **Member 8:** SQLAlchemy/psycopg database access.
- **Member 2:** tests against a real database.

Our small models and tests support the routing lesson; they do not replace those teammates' work.

**Harbour is separate from Library Loans API.** Apply routing ideas to Harbour's agreed API contract, not by copying book/loan resources into it. Harbour's Week 4 milestone is a core API skeleton with SQL, tests and Docker.

**Presentation sequence:** Explain request anatomy → run book routes → run loan routes → inspect failures → serve with Uvicorn → build one route independently.

## 2. Setup on Windows with WSL

Run in a **WSL terminal**, not a Python cell:
```bash
mkdir -p ~/harbour-learning/week4
cd ~/harbour-learning/week4
python3 -m venv .venv
source .venv/bin/activate
python -m pip install fastapi uvicorn httpx jupyterlab
jupyter lab
```
Put this notebook and the supplied `app.py` in that folder. Open the notebook using the Jupyter URL, or select this WSL virtual environment as your VS Code kernel.

**Two ways to run the lesson:**
1. Notebook cells build `app`; `TestClient` checks it without starting a server.
2. `app.py` contains the same routes for Uvicorn and `/docs`.

No PostgreSQL is required for this routing prototype. For a repeatable team environment, agree on dependency versions and commit a lockfile in the project.

## 3. A route is a method plus a path

A **client** sends an HTTP request; the API returns a status code and possibly a body.

```http
GET /books/7 HTTP/1.1
```

```python
@app.get("/books/{book_id}")
def get_book(book_id: int):
    ...
```
- `GET`: requested operation.
- `/books/{book_id}`: route path template.
- `book_id`: a variable section of the path.
- The function is the **handler**: FastAPI calls it when the method and path match.
- The decorator registers the route; the function name does not determine its URL.

`GET /books` and `POST /books` can share the same path because their methods differ. Declared routes do not automatically implement CRUD—you write each handler.

## 4. Path, query and body: three different places

| Place | Example | Purpose in our API |
|---|---|---|
| Path parameter | `/books/7` → `book_id=7` | Select a specific resource. |
| Query parameter | `/books?q=map&limit=5` | Filter or control a collection response. |
| JSON body | POST `/books` with `{"title":"Map","author":"Ada Fiction"}` | Supply structured creation/update data. |

A query string starts after `?`; `&` separates parameters. It is not part of the route template.

In our code:
```python
@app.get("/books/{book_id}")
def get_book(book_id: int): ...

@app.get("/books")
def list_books(q: str | None = None, limit: int = 10): ...
```
Names present in `{...}` come from the path. Other simple typed parameters normally come from the query. A Pydantic model parameter normally describes a body. We use `Path` and `Query` explicitly to add constraints.

Path parameters are required. Query parameters may be required or optional: a default makes them optional. They are not inherently optional.

## 5. Predict before coding

For `GET /books/12?preview=true`:
1. Method? **GET**.
2. Path parameter? **12**, if a route declares `/books/{book_id}`.
3. Query parameter? **preview=true**.

For `GET /books?limit=3`, write the collection route as `/books`, **not** `/books?limit={limit}`.

**Boundary case:** `/books/abc` matches the path shape, but cannot satisfy `book_id: int`. FastAPI returns **422**. `/books/999` has a valid integer ID, but our handler returns **404** if that book is absent.

**Routing order:** Declare fixed routes such as `/books/search` before `/books/{book_id}`. Otherwise `search` may be captured as the ID and fail validation. Query filtering on `/books?q=...` avoids needing that extra fixed route here.

In [1]:
from datetime import date
from typing import Annotated, Literal
from fastapi import FastAPI, HTTPException, Path, Query, Response, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field

print("Imports ready. TestClient sends requests directly to the ASGI application.")

Imports ready. TestClient sends requests directly to the ASGI application.


In [2]:
app = FastAPI(title="Library Loans Routing Lesson", version="0.1.0")

books: dict[int, dict] = {}
loans: dict[int, dict] = {}
members = {1: "Ada Example", 2: "Boma Example"}  # Fictional, pre-registered members.
next_book_id = 1
next_loan_id = 1


def reset_demo():
    """Learning/test utility, not a public route."""
    global next_book_id, next_loan_id
    books.clear()
    loans.clear()
    next_book_id = 1
    next_loan_id = 1


@app.get("/health")
def health():
    return {"status": "ok"}

## 6. Small request models keep JSON bodies clear

`BookCreate` describes JSON for a new book; we also use it for a full editable-field replacement with PUT. `BookRead` describes the returned record including its generated ID.

`LoanCreate` takes a book ID, a seeded member ID and a date. `LoanPatch` permits changes to due date or loan status; omission means “leave unchanged”.

**Validation boundary:** We enforce positive IDs, nonblank titles/authors, date parsing and allowed status labels. ISBN format, future-date policies, fines, physical-copy availability and complete borrowing rules remain for the service/validation owners.

Our loan representation references a catalogue book for simplicity. The final schema may lend a **physical copy** instead; agree that with the schema owner before integrating.

In [3]:
class BookCreate(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")
    title: str = Field(min_length=1)
    author: str = Field(min_length=1)


class BookRead(BookCreate):
    book_id: int


class LoanCreate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    book_id: int = Field(gt=0)
    member_id: int = Field(gt=0)
    due_on: date


class LoanPatch(BaseModel):
    model_config = ConfigDict(extra="forbid")
    due_on: date | None = None
    status: Literal["active", "returned"] | None = None


class LoanRead(LoanCreate):
    loan_id: int
    status: Literal["active", "returned"]

## 7. Agree on the endpoint map

| Operation | Books | Loans | Success |
|---|---|---|---|
| Create | POST `/books` | POST `/loans` | 201 + record |
| Read collection | GET `/books?q=map&offset=0&limit=10` | GET `/loans?member_id=1&loan_status=active` | 200 + list, possibly empty |
| Read one | GET `/books/{book_id}` | GET `/loans/{loan_id}` | 200 + record |
| Update | PUT `/books/{book_id}` | PATCH `/loans/{loan_id}` | 200 + updated record |
| Delete | DELETE `/books/{book_id}` | DELETE `/loans/{loan_id}` | 204, no body |

**Demo policies:** A referenced book cannot be deleted. Active loans cannot be deleted; mark returned first. Returned loans cannot be edited again.

**Contract warning:** Loan deletion is included to teach all CRUD operations because the assignment requests CRUD. A real library may prohibit hard deletion of *all* loans to preserve history, as a previous draft proposed. Agree on the final contract before merging; do not silently enable historical deletion.

PUT replaces all editable book fields here and returns 404 for a missing book. PATCH changes only supplied loan fields. A create-on-PUT policy is possible but not our contract.

In [4]:
BookId = Annotated[int, Path(gt=0)]
LoanId = Annotated[int, Path(gt=0)]


@app.post("/books", response_model=BookRead, status_code=201)
def create_book(payload: BookCreate, response: Response):
    global next_book_id
    book_id = next_book_id
    next_book_id += 1
    record = {"book_id": book_id, **payload.model_dump()}
    books[book_id] = record
    response.headers["Location"] = f"/books/{book_id}"
    return record


@app.get("/books", response_model=list[BookRead])
def list_books(
    q: Annotated[str | None, Query(min_length=1)] = None,
    offset: Annotated[int, Query(ge=0)] = 0,
    limit: Annotated[int, Query(ge=1, le=100)] = 10,
):
    records = sorted(books.values(), key=lambda record: record["book_id"])
    if q is not None:
        records = [record for record in records if q.casefold() in record["title"].casefold()]
    return records[offset:offset + limit]

## 8. Read the collection handler line by line

For `/books?q=map&offset=0&limit=5`:
1. FastAPI parses the query into `q="map"`, `offset=0`, `limit=5`.
2. The type annotations and Query rules validate them.
3. We order records by book ID for a predictable demo response.
4. If `q` was supplied, we keep titles containing that text, ignoring case.
5. The slice returns at most five records, beginning at offset zero.

Omit all parameters → defaults apply: no filter, offset 0, limit 10.

`?limit=0` fails validation with 422. No matching titles returns **200 with `[]`**, not 404: the collection route exists, but the filtered result is empty.

**Production handoff:** This demo scans and sorts memory. Push filtering/pagination into the repository's database query for the real app; these Python operations are not a performance model for PostgreSQL.

In [5]:
def require_book(book_id: int):
    if book_id not in books:
        raise HTTPException(status_code=404, detail="Book not found")
    return books[book_id]


@app.get("/books/{book_id}", response_model=BookRead)
def get_book(book_id: BookId):
    return require_book(book_id)


@app.put("/books/{book_id}", response_model=BookRead)
def replace_book(book_id: BookId, payload: BookCreate):
    require_book(book_id)
    books[book_id] = {"book_id": book_id, **payload.model_dump()}
    return books[book_id]


@app.delete("/books/{book_id}", status_code=204)
def delete_book(book_id: BookId):
    require_book(book_id)
    if any(loan["book_id"] == book_id for loan in loans.values()):
        raise HTTPException(status_code=409, detail="Book is referenced by a loan")
    del books[book_id]
    return Response(status_code=204)

In [6]:
reset_demo()
client = TestClient(app)
assert client.get("/health").json() == {"status": "ok"}

created = client.post("/books", json={"title": "The Lantern Map", "author": "Ada Fiction"})
assert created.status_code == 201
book_id = created.json()["book_id"]
assert created.headers["location"] == f"/books/{book_id}"
assert client.get(f"/books/{book_id}").json()["title"] == "The Lantern Map"
assert len(client.get("/books", params={"q": "MAP", "limit": 1}).json()) == 1
assert client.get("/books", params={"q": "absent"}).json() == []

updated = client.put(f"/books/{book_id}", json={"title": "Revised Map", "author": "Ada Fiction"})
assert updated.status_code == 200
assert updated.json()["title"] == "Revised Map"
deleted = client.delete(f"/books/{book_id}")
assert deleted.status_code == 204
assert deleted.content == b""
assert client.get(f"/books/{book_id}").status_code == 404
print("PASS: create, collection query, read, replace and delete a book.")

PASS: create, collection query, read, replace and delete a book.


## 9. Loans combine path/query routing with references

Create a loan with a JSON body:
```json
{"book_id": 1, "member_id": 1, "due_on": "2027-01-20"}
```
`book_id` here is a **body field**, not a path parameter. Its name alone does not decide where it comes from.

- `GET /loans/3`: loan ID is in the **path**.
- `GET /loans?member_id=1`: member filter is in the **query**.
- `PATCH /loans/3` with `{"status":"returned"}`: ID is in the **path**, change is in the **body**.

Our prototype rejects missing book/member references and duplicate active loans for the same book/member combination. It does not model available physical copies or protect against concurrent borrowers. Move those business rules into the service layer with transactional database enforcement.

In [7]:
@app.post("/loans", response_model=LoanRead, status_code=201)
def create_loan(payload: LoanCreate, response: Response):
    global next_loan_id
    require_book(payload.book_id)
    if payload.member_id not in members:
        raise HTTPException(status_code=404, detail="Member not found")
    if any(record["book_id"] == payload.book_id
           and record["member_id"] == payload.member_id
           and record["status"] == "active" for record in loans.values()):
        raise HTTPException(status_code=409, detail="This member already has an active loan for this book")
    loan_id = next_loan_id
    next_loan_id += 1
    record = {"loan_id": loan_id, **payload.model_dump(), "status": "active"}
    loans[loan_id] = record
    response.headers["Location"] = f"/loans/{loan_id}"
    return record


@app.get("/loans", response_model=list[LoanRead])
def list_loans(
    member_id: Annotated[int | None, Query(gt=0)] = None,
    loan_status: Annotated[Literal["active", "returned"] | None, Query()] = None,
    offset: Annotated[int, Query(ge=0)] = 0,
    limit: Annotated[int, Query(ge=1, le=100)] = 10,
):
    records = sorted(loans.values(), key=lambda record: record["loan_id"])
    if member_id is not None:
        records = [record for record in records if record["member_id"] == member_id]
    if loan_status is not None:
        records = [record for record in records if record["status"] == loan_status]
    return records[offset:offset + limit]

In [8]:
def require_loan(loan_id: int):
    if loan_id not in loans:
        raise HTTPException(status_code=404, detail="Loan not found")
    return loans[loan_id]


@app.get("/loans/{loan_id}", response_model=LoanRead)
def get_loan(loan_id: LoanId):
    return require_loan(loan_id)


@app.patch("/loans/{loan_id}", response_model=LoanRead)
def update_loan(loan_id: LoanId, payload: LoanPatch):
    record = require_loan(loan_id)
    changes = payload.model_dump(exclude_unset=True)
    # Omitted fields stay unchanged. Explicit null is not an allowed change.
    if not changes or any(value is None for value in changes.values()):
        raise HTTPException(status_code=422, detail="Supply at least one non-null editable field")
    if record["status"] == "returned":
        raise HTTPException(status_code=409, detail="Returned loans cannot be edited in this demo")
    loans[loan_id] = {**record, **changes}
    return loans[loan_id]


@app.delete("/loans/{loan_id}", status_code=204)
def delete_loan(loan_id: LoanId):
    record = require_loan(loan_id)
    if record["status"] == "active":
        raise HTTPException(status_code=409, detail="Return the loan before deleting it in this demo")
    del loans[loan_id]
    return Response(status_code=204)

## 10. Understand PATCH without losing existing fields

```python
changes = payload.model_dump(exclude_unset=True)
```
This keeps only fields actually supplied by the client.

For `{"status":"returned"}`, `due_on` is omitted, so we preserve the existing due date. Explicit `{"due_on":null}` is different from omission; our handler rejects it with 422.

```python
loans[loan_id] = {**record, **changes}
```
Build a new record from the old values, then replace only the supplied fields.

**Policy detail:** A second return PATCH receives 409 under this demo's returned-record edit policy. The resource state stays returned. Identical response codes are not required for an operation to have an idempotent final effect; PATCH in general is not guaranteed idempotent.

**Ask the class:** Would replacing the entire record with `changes` be correct? No: it would discard IDs and the fields the client did not supply.

In [9]:
reset_demo()
new_book = client.post("/books", json={"title": "Quiet Harbour", "author": "Boma Fiction"})
book_id = new_book.json()["book_id"]
loan_body = {"book_id": book_id, "member_id": 1, "due_on": "2027-01-20"}
created = client.post("/loans", json=loan_body)
assert created.status_code == 201
loan_id = created.json()["loan_id"]
assert created.headers["location"] == f"/loans/{loan_id}"
assert client.get(f"/loans/{loan_id}").status_code == 200
filtered = client.get("/loans", params={"member_id": 1, "loan_status": "active"})
assert len(filtered.json()) == 1
assert client.delete(f"/loans/{loan_id}").status_code == 409
assert client.delete(f"/books/{book_id}").status_code == 409
assert client.post("/loans", json=loan_body).status_code == 409

changed = client.patch(f"/loans/{loan_id}", json={"due_on": "2027-01-25"})
assert changed.status_code == 200
assert changed.json()["status"] == "active"
returned = client.patch(f"/loans/{loan_id}", json={"status": "returned"})
assert returned.json()["due_on"] == "2027-01-25"
deleted_loan = client.delete(f"/loans/{loan_id}")
assert deleted_loan.status_code == 204
assert deleted_loan.content == b""
assert client.get(f"/loans/{loan_id}").status_code == 404
assert client.delete(f"/books/{book_id}").status_code == 204
print("PASS: loan CRUD, filters, reference protection and duplicate conflict.")

PASS: loan CRUD, filters, reference protection and duplicate conflict.


In [10]:
# Break one requirement at a time. Validate actual responses, not guesses.
reset_demo()
book_id = client.post("/books", json={"title": "Map", "author": "Ada Fiction"}).json()["book_id"]
loan_id = client.post("/loans", json={"book_id": book_id, "member_id": 1,
                                      "due_on": "2027-01-20"}).json()["loan_id"]
failures = [
    ("Non-integer path", client.get("/books/not-an-int"), 422),
    ("Non-positive path", client.get("/books/0"), 422),
    ("Missing book", client.get("/books/99999"), 404),
    ("Invalid query type", client.get("/books?limit=hello"), 422),
    ("Query below minimum", client.get("/books?limit=0"), 422),
    ("Blank title", client.post("/books", json={"title": "  ", "author": "Ada Fiction"}), 422),
    ("Incomplete PUT", client.put(f"/books/{book_id}", json={"title": "Only a title"}), 422),
    ("Invalid due-date format", client.post("/loans", json={"book_id": book_id,
        "member_id": 1, "due_on": "not-a-date"}), 422),
    ("Missing member reference", client.post("/loans", json={"book_id": book_id,
        "member_id": 99999, "due_on": "2027-01-20"}), 404),
    ("Invalid loan query status", client.get("/loans?loan_status=lost"), 422),
    ("Explicit null PATCH", client.patch(f"/loans/{loan_id}", json={"due_on": None}), 422),
    ("Empty PATCH", client.patch(f"/loans/{loan_id}", json={}), 422),
]
for label, response, expected in failures:
    assert response.status_code == expected, (label, response.text)
    print(f"PASS: {label} -> {expected}")
print("Example validation detail:", failures[0][1].json()["detail"][0]["loc"])
reset_demo()  # Leave the application empty for independent practice.

PASS: Non-integer path -> 422
PASS: Non-positive path -> 422
PASS: Missing book -> 404
PASS: Invalid query type -> 422
PASS: Query below minimum -> 422
PASS: Blank title -> 422
PASS: Incomplete PUT -> 422
PASS: Invalid due-date format -> 422
PASS: Missing member reference -> 404
PASS: Invalid loan query status -> 422
PASS: Explicit null PATCH -> 422
PASS: Empty PATCH -> 422
Example validation detail: ['path', 'book_id']


## 11. FastAPI versus Uvicorn

- **FastAPI** defines the ASGI application, routes, validation and OpenAPI documentation.
- **Uvicorn** is the server that accepts network connections and runs that application.
- **TestClient** calls the app in-process for tests; it does not prove Uvicorn is running or that your browser can reach it.

From the WSL folder containing the supplied `app.py`, run in a terminal:
```bash
source .venv/bin/activate
python -m uvicorn app:app --host 127.0.0.1 --port 8000 --reload
```
`app:app` means **module `app.py` : variable `app`**. Start from the folder where Python can import that module.

Open in your Windows browser:
- `http://localhost:8000/health`
- `http://localhost:8000/docs` — interactive Swagger UI
- `http://localhost:8000/openapi.json` — machine-readable contract

Stop with **Ctrl+C**. `--reload` is for development; restarting/reloading erases this demo's in-memory data.

If localhost forwarding does not work in your WSL setup, investigate WSL networking and firewall settings. Bind to `0.0.0.0` only when required and understand which networks can then reach your unauthenticated demo. `0.0.0.0` is a bind address, not a browser destination.

## 12. Live demonstration with /docs

1. Start Uvicorn using `app.py`. Notebook dictionaries and server dictionaries are separate processes; data created in notebook tests will not appear in the server.
2. POST `/books`: select **Try it out**, enter `{"title":"The Lantern Map","author":"Ada Fiction"}`, execute.
3. Record the generated `book_id`; expect 201 and a Location header.
4. GET `/books/{book_id}` using that ID; expect 200.
5. GET `/books` with `q=lantern` and `limit=1`; explain why these fields are query parameters.
6. POST `/loans` using that book ID, member 1 and a valid ISO date.
7. PATCH the generated loan ID with `{"status":"returned"}`.
8. Try a bad path ID or invalid limit; explain 422 versus a missing-record 404.

### Equivalent WSL terminal requests
```bash
curl -i http://localhost:8000/health
curl -i -X POST http://localhost:8000/books \
  -H 'Content-Type: application/json' \
  -d '{"title":"The Lantern Map","author":"Ada Fiction"}'
curl -i 'http://localhost:8000/books?q=lantern&limit=1'
```
Quote URLs containing `&` so your shell does not treat it as a background operator. Let clients encode query values containing spaces or special characters; TestClient's `params={...}` does that for us.

## 13. Transfer the idea to Harbour

**Teaching proposal only:** If Harbour's agreed contract includes documents:
- `GET /documents/{document_id}` could identify one document with a path parameter.
- `GET /documents?limit=10&offset=0` could control a collection through query parameters.
- `POST /documents` could accept structured document metadata/content in a JSON body, depending on the agreed ingestion design.

Do not put full private document text, credentials or API keys into query strings; URLs can appear in logs and history. Choose a body and the appropriate authentication/security controls.

**Integration boundary:** Router parses HTTP → service enforces business rules → repository accesses PostgreSQL → router returns the agreed response. Our single-file dictionary implementation compresses these layers for learning, not as a recommended production layout.

**Friday handoff:** Agree schemas and errors with Members 1, 7 and 8. Replace dictionaries with the repository/session implementation, preserve the HTTP contract, and rerun Member 2's database-backed tests. In-memory passing tests do not demonstrate persistence, transactions, authentication or concurrency safety.

## 14. Independent exercise: implement a route, not just explain one

### A — Add a book query
Extend GET `/books` with optional `author: str | None`.
- Match the author's whole label case-insensitively.
- With both `q` and `author`, require both filters to match.
- Leave pagination defaults unchanged.

Write expected responses before editing: two books by Ada Fiction, one by Boma Fiction; test no filter, Ada filter, combined filters with no match, and invalid limit.

**Route-registration warning:** Edit the existing handler and rebuild the app by rerunning its definition cells from a fresh kernel, or restart Uvicorn after changing `app.py`. Adding a second handler for the same method/path is not a reliable way to replace the first route.

### B — Extend a loan query
Add an optional `due_before: date` query to GET `/loans`. Define whether the boundary is exclusive; for this exercise use `due_on < due_before`. Test a date before, equal to, and after the boundary.

### C — Explain without a screen
For `PATCH /loans/4` with `{"status":"returned"}`, name each value's source and describe one success and one conflict.

For one task, record **Restate → Examples → Plan → Test → Reflect**. Then explain the change live before merge.

In [11]:
# Learner workspace: write expectations and a test here before editing app.py.
# A: fixtures needed, expected author filter, expected combined filter, invalid input.
# B: define the boundary and write expected responses for equal/before/after dates.
# C: path value = ...; body value = ...; success = ...; conflict = ...

# Example assertion structure after implementing A:
# response = client.get("/books", params={"author": "Ada Fiction", "limit": 2})
# assert response.status_code == 200
# assert all(book["author"].casefold() == "ada fiction" for book in response.json())
# Also assert an expected count: all(...) alone passes on an empty list.

## 15. Twenty question-round prompts

Ask for the answer **and a reason**. Hide the next cell's answers during presentation.

1. What two things identify a route?
2. In GET `/books/7`, where does 7 come from?
3. In GET `/books?limit=5`, where does 5 come from?
4. Should the decorator contain `?limit={limit}`?
5. Must all query parameters be optional?
6. What makes our `limit` query optional?
7. Where do new-book title and author come from?
8. What happens for `/books/not-an-int`?
9. How is a valid but absent book ID different?
10. What should an empty filtered collection return?
11. Why can GET and POST share `/books`?
12. What does PUT require in our book contract?
13. What does `exclude_unset=True` preserve in a PATCH?
14. Does explicit null mean the same thing as omission here?
15. Why does loan creation return 201?
16. What is the response body for a successful 204 deletion?
17. Why register `/books/search` before `/books/{book_id}`?
18. What does `uvicorn app:app` refer to?
19. Does TestClient start a network server or test PostgreSQL persistence?
20. Why is deleting a returned loan a demo policy, not automatically a production requirement?

## 16. Answer cues and exercise guidance

1. HTTP method + path. 2. Path parameter. 3. Query parameter. 4. No; route path is `/books`.
5. No; query parameters without defaults can be required. 6. Its default value is 10.
7. JSON request body described by BookCreate. 8. 422 validation error. 9. 404 from the handler.
10. 200 with `[]`. 11. They have different methods. 12. All editable fields: title and author.
13. Fields not supplied stay unchanged. 14. No; explicit null is rejected in our PATCH.
15. It creates a new resource. 16. No body. 17. Prevent `search` being captured as an ID.
18. Module `app.py`, variable `app`. 19. No to both; it calls this in-memory app in-process.
20. Loan history retention is a team contract decision; the demo deliberately exposes CRUD for teaching.

**A guidance:** Filter the records before slicing; combine the author condition with the title condition. Include expected counts so an empty response does not accidentally pass your test.

**B guidance:** Add a typed optional query parameter. Compare stored date objects to the parsed boundary date. Invalid dates should produce 422; a valid date with no matching loans should produce 200 with `[]`.

**C guidance:** 4 is the path ID; returned is a body value. An active existing loan can return 200; trying to edit an already returned loan yields 409 under our contract.

In [12]:
# Verify the declared contract independently of individual successful calls.
schema = client.get("/openapi.json").json()
assert "get" in schema["paths"]["/books"]
assert "post" in schema["paths"]["/books"]
path_parameters = schema["paths"]["/books/{book_id}"]["get"]["parameters"]
assert any(p["name"] == "book_id" and p["in"] == "path" and p["required"]
           for p in path_parameters)
query_parameters = schema["paths"]["/books"]["get"]["parameters"]
assert any(p["name"] == "limit" and p["in"] == "query" and not p["required"]
           for p in query_parameters)
print("PASS: OpenAPI identifies the required path ID and optional query limit.")
print("API paths:", sorted(schema["paths"]))
client.close()

PASS: OpenAPI identifies the required path ID and optional query limit.
API paths: ['/books', '/books/{book_id}', '/health', '/loans', '/loans/{loan_id}']


## 17. Exit ticket and explain-before-merge checklist

Without copying, write:
1. A route for one book selected by a positive integer path ID.
2. A collection route with an optional integer `limit` defaulting to 10.
3. One request to each route, and why a value belongs in the path or query.
4. The Uvicorn command that serves a FastAPI variable named `app` in `app.py`.

A reviewer should check:
- [ ] Method/path and parameter sources are explicit.
- [ ] Path, query and JSON-body validation are tested separately.
- [ ] Collection filters, boundary limits and empty results are tested.
- [ ] Create/read/update/delete behaviour and conflict rules are agreed.
- [ ] No-body deletion responses really have no body.
- [ ] Database and service-layer integration work is not claimed complete by this demo.
- [ ] The author can explain a deliberately failed request and a trade-off.

**Known limits:** No persistence, real member CRUD, physical-copy inventory, fines, auth, transaction/concurrency guarantees or production deployment setup. Multiple Uvicorn workers would have separate dictionary stores. Teaching examples are not a drop-in production API.

**AI disclosure:** This material was AI-assisted. Hand-write the route you own, explain it live, and disclose which tests you ran before merge.

**Sources:** [Harbour Week 4 plan](https://github.com/Snrboi/harbour/blob/learn/README.md#week-4-fastapi-validation-tests-docker-and-ci-mini-project-1), [FastAPI path parameters](https://fastapi.tiangolo.com/tutorial/path-params/), [query parameters](https://fastapi.tiangolo.com/tutorial/query-params/), [request bodies](https://fastapi.tiangolo.com/tutorial/body/), [testing](https://fastapi.tiangolo.com/tutorial/testing/), [Uvicorn settings](https://www.uvicorn.org/settings/).

**Takeaway:** Path identifies the resource; query controls the selection/options; body carries structured changes. The contract—not the variable name—decides what belongs where.